In [1]:
!pip install -U langchain langchain-openai langchain-community

  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
from google.colab import userdata
import os

# os.environ에 담는 이유
# 우리가 가져다 쓰는 각 llm provider사들의 라이브러리 함수들이 첫 시작으로 자식의 key값을 os.environ에서 찾는 것이 첫 시작이기 때문

# os.environ['LANGSMITH_TRACING'] = 'true'
# os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
# os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
# os.environ['LANGSMITH_PROJECT'] = "skn26_langchain"

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [3]:
import pandas as pd

# 1. CSV 읽기
df = pd.read_csv("menu.csv")

# 2. name + description 합쳐서 새로운 컬럼 생성
df['menu_desc'] = df['name'].fillna('') + ' ' + df['description'].fillna('')

# 3. 리스트로 변환
menu_desc = df['menu_desc'].tolist()

# 확인
print(menu_desc[:5])

['회전초밥 ', '장어초밥 ', '프레지에 1호 홀케이크 ', '프레지에 ', '페르디셔 압펠 ']


In [4]:
len(menu_desc)

2008

In [5]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
다음은 메뉴명과 해당 메뉴에 대한 설명, 그리고 가게 관련 정보가 함께 포함된 텍스트이다.

입력된 내용을 바탕으로:
- 메뉴의 특징, 맛, 구성 등을 자연스럽게 설명하고
- 설명에 포함된 이벤트, 혜택, 추가 정보(할인, 세트, 한정메뉴 등)도 함께 반영하여
- 전체 내용을 2~3문장으로 자연스럽게 요약하라

주의사항:
- 메뉴명은 반드시 문장의 앞부분에 자연스럽게 포함할 것
- 불필요한 반복 없이 간결하게 작성할 것
- 정보는 삭제하지 말고 자연스럽게 녹여낼 것

추가 규칙:
- 만약 상세 설명이 부족하거나 일부 정보가 없는 경우에도,
  메뉴명과 제공된 정보만을 활용하여 자연스럽고 구체적인 메뉴 설명을 생성할 것
- 설명이 없는 경우에는 일반적인 조리 방식, 맛, 특징 등을 합리적으로 보완하여 작성할 것
- 단, 과장되거나 허위 정보는 생성하지 말 것

입력:
{menu_desc}

출력:
""")

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)

chain = prompt | llm

# menu_desc 리스트 → LLM 입력 형태로 변환
inputs = [{"menu_desc": x} for x in menu_desc]

# 배치 처리
batch_size = 20
results = []

for i in range(0, len(inputs), batch_size):
    batch = inputs[i:i+batch_size]
    output = chain.batch(batch)
    results.extend([o.content.strip() for o in output])

# 결과 확인
print(results[:5])

df['prompted_description'] = results

['회전초밥은 신선한 재료로 만든 다양한 초밥을 회전 컨베이어 벨트 위에서 편리하게 즐길 수 있는 메뉴로, 합리적인 가격과 빠른 서비스가 특징입니다. 가게에서는 특정 시간대 할인이나 세트 메뉴 등 다양한 혜택을 제공해 더욱 부담 없이 맛있는 초밥을 경험할 수 있습니다.', '장어초밥은 부드럽고 고소한 장어가 신선한 초밥 위에 올려져 깊은 풍미를 자랑하며, 특별한 소스가 더해져 감칠맛을 더합니다. 가게에서는 장어초밥 구매 시 다양한 이벤트와 할인 혜택을 제공해 더욱 합리적으로 즐길 수 있습니다.', '프레지에 1호 홀케이크는 신선한 재료로 정성껏 만든 부드럽고 촉촉한 케이크로, 특별한 날을 위한 완벽한 선택입니다. 다양한 이벤트와 할인 혜택이 함께 제공되어 더욱 합리적으로 즐길 수 있습니다.', '프레지에는 부드러운 크림과 신선한 과일이 조화를 이루는 디저트로, 상큼하면서도 달콤한 맛이 특징입니다. 현재 한정 이벤트로 세트 구매 시 할인 혜택이 제공되어 더욱 합리적으로 즐길 수 있습니다.', '페르디셔 압펠은 신선한 사과의 상큼한 맛과 부드러운 식감이 조화를 이루는 메뉴로, 가게에서 특별한 이벤트와 함께 즐길 수 있습니다. 한정 수량으로 제공되며, 할인 혜택이나 세트 구성도 마련되어 있어 더욱 합리적으로 맛볼 수 있습니다.']


In [7]:
print(len(df), len(results))

2008 2008


In [8]:
import pandas as pd
import numpy as np
from openai import OpenAI

client = OpenAI()

# 1. 임베딩 함수
def get_embeddings(text_list):
    response = client.embeddings.create(
        model="text-embedding-3-small",  # 가볍고 충분히 성능 좋음
        input=text_list
    )
    return [e.embedding for e in response.data]


# 2. BLOB 변환 함수 (list → bytes)
def to_blob(embedding):
    return np.array(embedding, dtype=np.float32).tobytes()


# 3. 배치 임베딩
batch_size = 50
all_embeddings = []

texts = df['prompted_description'].tolist()

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    emb = get_embeddings(batch)
    all_embeddings.extend(emb)


# 4. df에 벡터 그대로 저장 (list 형태)
df['embedding_blobX'] = all_embeddings

In [9]:
import numpy as np
import base64

# 1. embedding 리스트 가져오기
embedding_list = df['embedding_blobX'].tolist()   # list of list

# 2. list → blob → base64 인코딩 (한 번에 처리)
encoded_list = [
    base64.b64encode(
        np.array(vec, dtype=np.float32).tobytes()
    ).decode('utf-8')
    for vec in embedding_list
]

# 3. df에 저장
df['embedding'] = encoded_list

In [10]:
df.to_csv("menu.csv", index=False, encoding="utf-8-sig")

In [13]:
df['embedding'][:2]

,embedding
0,AOA7PADgKL0AYNS7ACAgPACAnzwAAO67AAC8uwBA5zwAII...
1,AIDsuwCghb0AoBe8AEBdPAAAZD0AAKq8AEBQuwDgET0AYP...


In [14]:
df = pd.read_csv("food.csv")
df['embedding'][:2]

,embedding
0,b'\x00\x80\xbe;\x00\xc0p\xbd\x00\xe0\x01\xbd\x...
1,b'\x00\x00\x0e<\x00\xe0b\xbd\x00`\x9f\xbd\x00\...
